In [54]:
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

In [55]:
lista = [i for i in range(1,10)]

dicio = {i:i for i in lista}

In [56]:
model = pyo.ConcreteModel()

model.set_p = pyo.Set(initialize=lista)
model.s = pyo.Var(model.set_p, within=NonNegativeReals)
model.y = pyo.Var(model.set_p,domain=pyo.Binary)
model.p = pyo.Var(model.set_p, within=NonNegativeReals)
model.profit = pyo.Param(model.set_p,initialize=dicio)
#Restrições
# Se y3 = 1 entao y1 e y2 também precisam ser 1
def restricao1(model):
    return model.y[3] <= model.y[1]
model.restricao1 = pyo.Constraint(rule=restricao1)
def restricao2(model):
    return model.y[3] <= model.y[2]
model.restricao2 = pyo.Constraint(rule=restricao2)

def restricao3(model):
    return model.y[6] + model.y[7] + model.y[8] <= 2
model.restricao3 = pyo.Constraint(rule=restricao3)

def restricao4(model):
    return model.y[3] + model.y[4] + 2*model.y[5] <= 2
model.restricao4 = pyo.Constraint(rule=restricao4)

# objetivo

def objetivo(model):
    return sum(model.profit[i]*model.y[i] for i in model.set_p)
model.objetivo = pyo.Objective(rule=objetivo, sense=maximize)


In [57]:
model.pprint()

1 Set Declarations
    set_p : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    9 : {1, 2, 3, 4, 5, 6, 7, 8, 9}

1 Param Declarations
    profit : Size=9, Index=set_p, Domain=Any, Default=None, Mutable=False
        Key : Value
          1 :     1
          2 :     2
          3 :     3
          4 :     4
          5 :     5
          6 :     6
          7 :     7
          8 :     8
          9 :     9

3 Var Declarations
    p : Size=9, Index=set_p
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          1 :     0 :  None :  None : False :  True : NonNegativeReals
          2 :     0 :  None :  None : False :  True : NonNegativeReals
          3 :     0 :  None :  None : False :  True : NonNegativeReals
          4 :     0 :  None :  None : False :  True : NonNegativeReals
          5 :     0 :  None :  None : False :  True : NonNegativeReals
          6 :     0 :  None :  None : False :  True : Non

In [58]:
# ------------------- solver
opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
res = opt.solve(model,tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer Community Edition 22.2.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2026.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\Pichau\AppData\Local\Temp\tmpaa862rth.cplex.log' open.
CPLEX> Problem 'C:\Users\Pichau\AppData\Local\Temp\tmp3z6i9paq.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\Pichau\AppData\Local\Temp\tmp3z6i9paq.pyomo.lp
Objective sense      : Maximize
Variables            :       9  [Binary: 9]
Objective nonzeros   :       9
Linear constraints   :       4  [Less: 4]
  Nonzeros           :      10
  RHS nonzeros       :       2

Variables            : Min LB: 0.000000         Max UB: 1.000000       
Objective nonzeros   : Min   : 1.0

In [59]:
for a in model.set_p:
    print(f'Ativo {a}: {model.y[a].value:.4f}')

Ativo 1: 1.0000
Ativo 2: 1.0000
Ativo 3: 1.0000
Ativo 4: 1.0000
Ativo 5: 0.0000
Ativo 6: 0.0000
Ativo 7: 1.0000
Ativo 8: 1.0000
Ativo 9: 1.0000


### 8.2

In [80]:
total = 18

In [60]:
colunas = [f'project_{i}' for i in range(1,11)]
periodos = [f'year_{i}' for i in range(1,5)]
npv = [30,30,20,15,15,15,15,24,18,18]

In [69]:
colunas

['project_1',
 'project_2',
 'project_3',
 'project_4',
 'project_5',
 'project_6',
 'project_7',
 'project_8',
 'project_9',
 'project_10']

In [61]:
periodos

['year_1', 'year_2', 'year_3', 'year_4']

In [62]:
df = pd.read_excel('1.xlsx').T
df

,0,1,2,3,4,5,6,7,8,9
30,30,30,20,15,15,15,15,24,18,18
12,12,0,3,10,0,0,0,8,0,0
4,4,12,4,0,11,0,0,8,0,0
4.1,4,4,4,0,0,12,0,0,10,4
0,0,4,4,0,0,0,13,0,4,10


In [63]:
df = df.reset_index(drop=True)
df

,0,1,2,3,4,5,6,7,8,9
0,30,30,20,15,15,15,15,24,18,18
1,12,0,3,10,0,0,0,8,0,0
2,4,12,4,0,11,0,0,8,0,0
3,4,4,4,0,0,12,0,0,10,4
4,0,4,4,0,0,0,13,0,4,10


In [64]:
df = df.rename(index={0:'npv'}).rename(index={i:periodos[i-1] for i in range(1,5)})


In [71]:
df = df.rename(columns={i:colunas[i] for i in range(0,10)})
df

,project_1,project_2,project_3,project_4,project_5,project_6,project_7,project_8,project_9,project_10
npv,30,30,20,15,15,15,15,24,18,18
year_1,12,0,3,10,0,0,0,8,0,0
year_2,4,12,4,0,11,0,0,8,0,0
year_3,4,4,4,0,0,12,0,0,10,4
year_4,0,4,4,0,0,0,13,0,4,10


In [105]:
df_custo = df.drop(index='npv')
df_custo.T.loc['project_2','year_1']

np.int64(0)

In [116]:
## sem restrições
model = pyo.ConcreteModel()

model.projetos = pyo.Set(initialize=colunas)
model.periodos = pyo.Set(initialize=periodos)
model.npv = pyo.Param(model.projetos,initialize=lambda model,p:df[df.index=='npv'].T.loc[p].values[0])
model.custo = pyo.Param(model.projetos,model.periodos,initialize=lambda model,p,pe:df_custo.T.loc[p,pe])
model.x = pyo.Var(model.projetos,model.periodos,domain=pyo.Binary)

In [117]:
model.pprint()

2 Set Declarations
    periodos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    4 : {'year_1', 'year_2', 'year_3', 'year_4'}
    projetos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :   10 : {'project_1', 'project_2', 'project_3', 'project_4', 'project_5', 'project_6', 'project_7', 'project_8', 'project_9', 'project_10'}

2 Param Declarations
    custo : Size=40, Index=projetos*periodos, Domain=Any, Default=None, Mutable=False
        Key                      : Value
         ('project_1', 'year_1') :    12
         ('project_1', 'year_2') :     4
         ('project_1', 'year_3') :     4
         ('project_1', 'year_4') :     0
        ('project_10', 'year_1') :     0
        ('project_10', 'year_2') :     0
        ('project_10', 'year_3') :     4
        ('project_10', 'year_4') :    10
         ('project_2', 'year_1') :     0
         ('p

In [126]:
# objetivo

def obj_r(model):
    return sum(sum(model.npv[p]*model.x[p,pe]  for p in model.projetos)for pe in model.periodos)
model.obj = pyo.Objective(rule=obj_r, sense=maximize)
# restricao custo total
def res(model):
    return sum(sum(model.custo[p,pe]*model.x[p,pe] for p in model.projetos) for pe in model.periodos) <= total
model.resres = pyo.Constraint(rule=res)

'pyomo.core.base.objective.ScalarObjective'>) on block unknown with a new
Component (type=<class 'pyomo.core.base.objective.AbstractScalarObjective'>).
This is usually indicative of a modelling error. To avoid this warning, use
block.del_component() and block.add_component().
'pyomo.core.base.constraint.ScalarConstraint'>) on block unknown with a new
Component (type=<class
'pyomo.core.base.constraint.AbstractScalarConstraint'>). This is usually
indicative of a modelling error. To avoid this warning, use
block.del_component() and block.add_component().


In [127]:
# ------------------- solver
opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
res = opt.solve(model,tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer Community Edition 22.2.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2026.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\Pichau\AppData\Local\Temp\tmp7ylexjkq.cplex.log' open.
CPLEX> Problem 'C:\Users\Pichau\AppData\Local\Temp\tmpmmjox0__.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\Pichau\AppData\Local\Temp\tmpmmjox0__.pyomo.lp
Objective sense      : Maximize
Variables            :      40  [Binary: 40]
Objective nonzeros   :      40
Linear constraints   :       1  [Less: 1]
  Nonzeros           :      20
  RHS nonzeros       :       1

Variables            : Min LB: 0.000000         Max UB: 1.000000       
Objective nonzeros   : Min   : 15

In [146]:
for pe in model.periodos:
    # print(f"Custo total do ano {pe}: {sum(pyo.value(model.custo[p,pe])*pyo.value(model.x[p,pe]) for p in model.projetos)}")

    for p in model.projetos:
        print(f'{p},{pe},"Custo: {pyo.value(model.custo[p,pe])}": {model.x[p,pe].value:.4f}')

project_1,year_1,"Custo: 12": 0.0000
project_2,year_1,"Custo: 0": 1.0000
project_3,year_1,"Custo: 3": 0.0000
project_4,year_1,"Custo: 10": 0.0000
project_5,year_1,"Custo: 0": 1.0000
project_6,year_1,"Custo: 0": 1.0000
project_7,year_1,"Custo: 0": 1.0000
project_8,year_1,"Custo: 8": 0.0000
project_9,year_1,"Custo: 0": 1.0000
project_10,year_1,"Custo: 0": 1.0000
project_1,year_2,"Custo: 4": 1.0000
project_2,year_2,"Custo: 12": 0.0000
project_3,year_2,"Custo: 4": 0.0000
project_4,year_2,"Custo: 0": 1.0000
project_5,year_2,"Custo: 11": 0.0000
project_6,year_2,"Custo: 0": 1.0000
project_7,year_2,"Custo: 0": 1.0000
project_8,year_2,"Custo: 8": 0.0000
project_9,year_2,"Custo: 0": 1.0000
project_10,year_2,"Custo: 0": 1.0000
project_1,year_3,"Custo: 4": 1.0000
project_2,year_3,"Custo: 4": 1.0000
project_3,year_3,"Custo: 4": 0.0000
project_4,year_3,"Custo: 0": 1.0000
project_5,year_3,"Custo: 0": 1.0000
project_6,year_3,"Custo: 12": 0.0000
project_7,year_3,"Custo: 0": 1.0000
project_8,year_3,"Cus